In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D10 — IMPI — Inquérito Mensal à Produção Industrial
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip -q install pymupdf pandas

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import re

import fitz
import pandas as pd

DOCUMENT_ID = "D10"
DOCUMENT_NAME = "IMPI — Inquérito Mensal à Produção Industrial"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "PyMuPDF native text-block conversion with "
    "source-coordinate ordering and page-aware Markdown"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5"
EXPECTED_PAGE_COUNT = 4

# ------------------------------------------------------------
# Frozen Stage 1 expectations
# Used ONLY after model extraction for diagnostics/validation.
# They are NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11
}

EXPECTED_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location"
]

STRING_OR_NULL_FIELDS = EXPECTED_FIELDS.copy()

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_QUESTIONNAIRE_CODES = {
    "BC001",
    "BC005",
    "BC007",
    "BC010",
    "BC015",
    "BC020",
    "BC025",
    "BC030"
}

OUTPUT_DIR = Path("outputs_D10_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D10_branch_B_source_diagnostics.json"
SOURCE_BLOCK_AUDIT_PATH = OUTPUT_DIR / "D10_branch_B_source_block_audit.csv"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D10_branch_B_structural_markdown.md"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D10_branch_B_conversion_integrity.json"
REPRESENTATION_PATH = OUTPUT_DIR / "D10_branch_B_representation.json"
PROMPT_PATH = OUTPUT_DIR / "D10_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D10_branch_B_experiment_metadata_pre.json"
RAW_RESPONSE_PATH = OUTPUT_DIR / "D10_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D10_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D10_branch_B_technical_diagnostics.json"
)
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D10_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D10_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Frozen Stage 1 records:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D10 PDF
# ============================================================

uploaded = files.upload()

pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError(
        "Upload exactly one original D10 PDF."
    )

SOURCE_PATH = pdf_paths[0]


def sha256_file(
    path,
    chunk_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Source identity
# ------------------------------------------------------------

SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError(
        "Unexpected D10 source format."
    )

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D10 PDF does not match "
        "the frozen Stage 1 source identity."
    )


# ------------------------------------------------------------
# Open source PDF and verify physical page count
# ------------------------------------------------------------

pdf_document = fitz.open(
    SOURCE_PATH
)

PAGE_COUNT = len(
    pdf_document
)

PAGE_COUNT_VALID = (
    PAGE_COUNT
    == EXPECTED_PAGE_COUNT
)

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; "
        f"observed {PAGE_COUNT}."
    )


# ------------------------------------------------------------
# Native text and page-level diagnostics
# ------------------------------------------------------------

page_rows = []
page_native_text = {}


for page_number, page in enumerate(
    pdf_document,
    start=1
):

    text = (
        page.get_text("text")
        or ""
    )

    page_native_text[
        page_number
    ] = text


    page_rows.append({

        "Page Number":
            page_number,

        "Native Character Count":
            len(text),

        "Native Word Count":
            len(
                text.split()
            ),

        "Text Block Count":
            len(
                page.get_text(
                    "blocks"
                )
            ),

        "Drawing Count":
            len(
                page.get_drawings()
            ),

        "Image Count":
            len(
                page.get_images(
                    full=True
                )
            ),

        "Widget Count":
            len(
                list(
                    page.widgets()
                    or []
                )
            ),

        "Width":
            float(
                page.rect.width
            ),

        "Height":
            float(
                page.rect.height
            ),

        "Rotation":
            int(
                page.rotation
            )
    })


page_diagnostics_df = pd.DataFrame(
    page_rows
)


# IMPORTANT:
# Use a real newline here, not the literal characters "\\n".
FULL_NATIVE_TEXT = "\n".join(
    page_native_text[
        page_number
    ]
    for page_number
    in range(
        1,
        PAGE_COUNT + 1
    )
)


TOTAL_NATIVE_CHARACTERS = len(
    FULL_NATIVE_TEXT
)

TEXT_EXTRACTABLE = (
    TOTAL_NATIVE_CHARACTERS
    > 100
)

OCR_REQUIRED = (
    not TEXT_EXTRACTABLE
)


if OCR_REQUIRED:
    raise ValueError(
        "D10 is expected to contain a usable "
        "native text layer; OCR is not part "
        "of Branch B for this document."
    )


# ------------------------------------------------------------
# Source-region marker checks
# ------------------------------------------------------------
#
# These are source-preservation diagnostics only.
# They do NOT contain Stage 1 expected answer values.
# ------------------------------------------------------------

SOURCE_MARKER_PATTERNS = {

    "survey_title":
        (
            r"IMPI\s*-\s*"
            r"Inquérito Mensal à Produção Industrial"
        ),

    "registration_number":
        r"\b10067\b",

    "validity_date":
        r"\b2026/12/31\b",

    "section_i":
        (
            r"Identificação da "
            r"unidade estatística"
        ),

    "section_ii":
        (
            r"Situação da unidade estatística"
        ),

    "section_iii":
        r"\bIII\s+Observações\b",

    "section_iv":
        (
            r"Responsável pelo "
            r"preenchimento"
        ),

    "uae":
        (
            r"Unidade de Atividade "
            r"Económica"
        ),

    "product_table":
        (
            r"QUANTIDADES\s+"
            r"PRODUZIDAS"
        ),

    "filling_instructions":
        r"INSTRUÇÕES DE PREENCHIMENTO",

    "explanatory_notes":
        r"NOTAS EXPLICATIVAS"
}


SOURCE_MARKER_STATUS = {

    name:
        bool(
            re.search(
                pattern,
                FULL_NATIVE_TEXT,
                flags=re.IGNORECASE
            )
        )

    for name, pattern
    in SOURCE_MARKER_PATTERNS.items()
}


ALL_SOURCE_MARKERS_PRESENT = all(
    SOURCE_MARKER_STATUS.values()
)


# ------------------------------------------------------------
# Printed questionnaire-code verification
# ------------------------------------------------------------

observed_source_codes = sorted(
    set(
        re.findall(
            r"\bBC\d{3}\b",
            FULL_NATIVE_TEXT
        )
    )
)


EXPECTED_SOURCE_CODES_PRESENT = (
    set(
        observed_source_codes
    )
    >= EXPECTED_QUESTIONNAIRE_CODES
)


# ------------------------------------------------------------
# Save source diagnostics
# ------------------------------------------------------------

SOURCE_DIAGNOSTICS = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "source_marker_status":
        SOURCE_MARKER_STATUS,

    "all_source_markers_present":
        ALL_SOURCE_MARKERS_PRESENT,

    "observed_questionnaire_codes":
        observed_source_codes,

    "expected_source_codes_present":
        EXPECTED_SOURCE_CODES_PRESENT
}


SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Display diagnostics
# ------------------------------------------------------------

print(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)

display(
    page_diagnostics_df
)


# ------------------------------------------------------------
# Stop only if the actual source check fails
# ------------------------------------------------------------

if not ALL_SOURCE_MARKERS_PRESENT:
    raise ValueError(
        "One or more expected D10 "
        "source regions are missing."
    )

if not EXPECTED_SOURCE_CODES_PRESENT:
    raise ValueError(
        "One or more expected printed D10 "
        "questionnaire codes are missing."
    )

## D10 Branch B conversion rule

D10 is already machine-readable, so Branch B does **not** introduce OCR. Instead, it converts the complete four-page PDF into an intermediate representation that makes the page sequence and native text-block organisation explicit.

For each physical page:

- every non-empty native text block is retained;
- blocks are ordered deterministically by their source coordinates;
- each block receives a structural Markdown heading and its original bounding-box coordinates;
- the original text inside the block is retained without semantic rewriting or value correction.

The representation intentionally retains the three repeated UAE blocks on page 2 and the sample product rows on page 3. Those are source content. Their treatment as one reusable template and as non-respondent examples is controlled by the fixed extraction task, not by deleting them from Branch B.

In [ ]:
# ============================================================
# 2. Create complete layout-aware structural Markdown
# ============================================================

def classify_block(block_text):
    """
    Deterministic structural label based only on visible source wording.
    The label is metadata; the block text itself is preserved unchanged.
    """

    compact = " ".join(str(block_text).split())
    upper = compact.upper()

    if "IMPI - INQUÉRITO MENSAL À PRODUÇÃO INDUSTRIAL" in upper:
        return "Document title"

    if re.search(r"\bI\s+Identificação da unidade estatística", compact):
        return "Section heading"

    if re.search(r"\bII\s+Situação da unidade estatística", compact):
        return "Section heading"

    if re.search(r"\bIII\s+Observações", compact):
        return "Section heading"

    if re.search(r"\bIV\s+Responsável pelo preenchimento", compact):
        return "Section heading"

    if "INSTRUÇÕES DE PREENCHIMENTO" in upper:
        return "Section heading"

    if "NOTAS EXPLICATIVAS" in upper:
        return "Section heading"

    if (
        "QUANTIDADES PRODUZIDAS" in upper
        or "QUANTIDADES VENDIDAS" in upper
        or "VALOR DAS VENDAS /" in upper
    ):
        return "Table region"

    if "UNIDADE DE ATIVIDADE ECONÓMICA (UAE)" in upper:
        return "UAE region"

    return "Source text block"


page_block_rows = []
page_markdown = {}

SOURCE_NONEMPTY_BLOCK_COUNT = 0

for page_number, page in enumerate(pdf_document, start=1):

    blocks = [
        block
        for block in page.get_text("blocks")
        if str(block[4]).strip()
    ]

    # Deterministic source-layout order:
    # top-to-bottom, then left-to-right.
    sorted_blocks = sorted(
        blocks,
        key=lambda block: (
            round(float(block[1]), 2),
            round(float(block[0]), 2)
        )
    )

    page_lines = []

    for block_index, block in enumerate(sorted_blocks, start=1):

        x0, y0, x1, y1, block_text, *_ = block

        retained_text = str(block_text).strip()

        SOURCE_NONEMPTY_BLOCK_COUNT += 1

        block_type = classify_block(retained_text)

        page_block_rows.append({
            "Page Number": page_number,
            "Block Index": block_index,
            "Block Type": block_type,
            "x0": float(x0),
            "y0": float(y0),
            "x1": float(x1),
            "y1": float(y1),
            "Character Count": len(retained_text),
            "Text": retained_text
        })

        page_lines.extend([
            (
                f"### Source Block {block_index:02d} — {block_type} "
                f"[bbox={x0:.1f},{y0:.1f},{x1:.1f},{y1:.1f}]"
            ),
            "",
            retained_text,
            ""
        ])

    page_markdown[page_number] = "\n".join(page_lines).rstrip()


markdown_parts = [
    "# D10 — IMPI Questionnaire",
    "",
    "> Complete layout-aware structural conversion of the original four-page PDF.",
    "> Every non-empty native text block is retained with its physical page and bounding-box metadata.",
    "> OCR, semantic rewriting, value correction and normalisation are not applied.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_parts.extend([
        f"## Source Page {page_number}",
        "",
        page_markdown[page_number],
        ""
    ])

STRUCTURAL_MARKDOWN = "\n".join(markdown_parts).rstrip() + "\n"

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

source_block_audit_df = pd.DataFrame(page_block_rows)

source_block_audit_df.to_csv(
    SOURCE_BLOCK_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Structural Markdown:", STRUCTURAL_MARKDOWN_PATH)
print("Representation SHA-256:", STRUCTURAL_MARKDOWN_SHA256)
print("Retained non-empty source blocks:", SOURCE_NONEMPTY_BLOCK_COUNT)
display(source_block_audit_df.head(30))


In [ ]:
# ============================================================
# 3. Conversion-integrity verification
# ============================================================

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}" in STRUCTURAL_MARKDOWN
    for page_number in range(1, EXPECTED_PAGE_COUNT + 1)
}

# Every retained source block must occur verbatim in the final
# structural representation.
source_block_preservation_checks = {}

for row_index, row in source_block_audit_df.iterrows():
    key = (
        f"page_{int(row['Page Number'])}"
        f"_block_{int(row['Block Index'])}"
    )

    source_block_preservation_checks[key] = (
        row["Text"] in STRUCTURAL_MARKDOWN
    )

all_source_blocks_preserved = all(
    source_block_preservation_checks.values()
)

CONVERSION_MARKER_PATTERNS = {
    "survey_title":
        r"IMPI\s*-\s*Inquérito Mensal à Produção Industrial",
    "registration_number":
        r"10067",
    "validity_date":
        r"2026/12/31",
    "section_i":
        r"Identificação da unidade estatística",
    "section_ii":
        r"Situação da unidade estatística",
    "section_iii":
        r"Observações",
    "section_iv":
        r"Responsável pelo preenchimento",
    "uae":
        r"Unidade de Atividade Económica",
    "product_table_nif":
        r"\bNIF\b",
    "product_table_uae":
        r"\bUAE\b",
    "product_table_period":
        r"Periodo de Referência|Período de Referência",
    "quantities_produced":
        r"QUANTIDADES\s+PRODUZIDAS",
    "quantities_sold":
        r"QUANTIDADES\s+VENDIDAS",
    "sales_value":
        r"VALOR DAS VENDAS",
    "filling_instructions":
        r"INSTRUÇÕES DE PREENCHIMENTO",
    "explanatory_notes":
        r"NOTAS EXPLICATIVAS",
    "prodcom":
        r"\bPRODCOM\b|3924/91"
}

conversion_marker_status = {
    marker: bool(
        re.search(
            pattern,
            STRUCTURAL_MARKDOWN,
            flags=re.IGNORECASE
        )
    )
    for marker, pattern
    in CONVERSION_MARKER_PATTERNS.items()
}

all_conversion_markers_preserved = all(
    conversion_marker_status.values()
)

converted_code_set = set(
    re.findall(
        r"\bBC\d{3}\b",
        STRUCTURAL_MARKDOWN
    )
)

questionnaire_codes_preserved = (
    converted_code_set
    >= EXPECTED_QUESTIONNAIRE_CODES
)

sample_rows_preserved_in_representation = all(
    marker in STRUCTURAL_MARKDOWN
    for marker in [
        "Produto a",
        "Produto b",
        "Produto c",
        "Produto d",
        "111111111111"
    ]
)

# The source visually repeats the UAE template three times.
# We do not deduplicate the representation.
uae_block_marker_count = len(
    re.findall(
        r"\[Codigo da UAE\]\s*-\s*\[Designação da UAE\]",
        STRUCTURAL_MARKDOWN,
        flags=re.IGNORECASE
    )
)

repeated_uae_content_retained = (
    uae_block_marker_count >= 3
)

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "native_text_characters": TOTAL_NATIVE_CHARACTERS,
    "ocr_required": OCR_REQUIRED,
    "ocr_applied": False,
    "source_nonempty_block_count": SOURCE_NONEMPTY_BLOCK_COUNT,
    "all_source_pages_retained": all(page_boundary_checks.values()),
    "page_boundary_checks": page_boundary_checks,
    "all_source_blocks_preserved": all_source_blocks_preserved,
    "source_block_preservation_checks": source_block_preservation_checks,
    "conversion_marker_status": conversion_marker_status,
    "all_conversion_markers_preserved": all_conversion_markers_preserved,
    "expected_questionnaire_codes_preserved": questionnaire_codes_preserved,
    "observed_questionnaire_codes": sorted(converted_code_set),
    "sample_product_rows_preserved_in_representation":
        sample_rows_preserved_in_representation,
    "uae_template_visual_instance_count":
        uae_block_marker_count,
    "repeated_uae_content_retained":
        repeated_uae_content_retained,
    "conversion_method":
        CONVERSION_METHOD,
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "scope_filtering_applied": False,
    "page_cropping_applied": False,
    "page_removal_applied": False,
    "table_value_reconstruction_applied": False,
    "form_value_reconstruction_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "source_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed": all([
        SOURCE_HASH_MATCH,
        PAGE_COUNT_VALID,
        TEXT_EXTRACTABLE,
        not OCR_REQUIRED,
        all(page_boundary_checks.values()),
        all_source_blocks_preserved,
        all_conversion_markers_preserved,
        questionnaire_codes_preserved,
        sample_rows_preserved_in_representation,
        repeated_uae_content_retained
    ])
}

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(CONVERSION_INTEGRITY, indent=2, ensure_ascii=False))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError(
        "D10 Branch B structural conversion failed integrity checks."
    )


In [ ]:
# ============================================================
# 4. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete PDF converted to page-aware layout-aware structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": EXPECTED_PAGE_COUNT,
    "structural_conversion_applied": True,
    "native_text_used_for_conversion": True,
    "source_block_geometry_preserved_as_metadata": True,
    "complete_source_document_retained": True,
    "page_boundaries_made_explicit": True,
    "conversion_method":
        CONVERSION_METHOD,
    "branch_name":
        BRANCH_NAME,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "scope_enforced_by_prompt_not_representation_filtering": True,
    "repeated_uae_blocks_retained": True,
    "sample_product_rows_retained": True,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "table_value_reconstruction_applied": False,
    "form_value_reconstruction_applied": False,
    "source_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(REPRESENTATION, indent=2, ensure_ascii=False))


## Controlled extraction task

The prompt below mirrors the final D10 Branch A extraction task. Only the representation-dependent wording and branch identifier change.

The model is **not told**:

- that 69 records are expected;
- the expected category distribution;
- the Stage 1 questionnaire-code assignments;
- Branch A or Validation A results.

The fixed scope still explicitly defines which questionnaire elements belong to the task, as it did in Branch A.

In [ ]:
# ============================================================
# 5. Create controlled Branch B extraction prompt
# ============================================================

BRANCH_B_PROMPT = """You are an information extraction assistant.

Extract the predefined structural and semantic questionnaire records
represented within the defined scope of the attached structurally
converted Markdown representation of:

IMPI — Inquérito Mensal à Produção Industrial.

Treat the attached structural Markdown representation as the only
source of information.

The supplied representation corresponds to the complete four physical
pages of the original questionnaire PDF.

The questionnaire is blank. Extract represented questionnaire
structure, labels, metadata, reusable template elements, table fields
and instructions. Do not invent or infer respondent answers.

For every included record return exactly these fields:

- Category
- Section
- Field or Concept
- Description
- Code
- Expected Value Type
- Source Location

Use exactly one of these Category values:

- Instrument metadata
- Questionnaire field
- UAE template element
- Product table field
- Instruction


1. Instrument metadata

From the header, legal notice and response-contact areas represented
from physical PDF page 1, extract one record for each of these
predefined concepts:

- Survey name
- Statistical system
- Legal basis
- INE registration number
- Validity date
- Electronic response URL
- Contact email
- Contact telephone

Read their represented content directly from the source representation.

Do not create separate records from decorative branding or graphical
elements.


2. Questionnaire fields

Extract the predefined response fields represented from physical PDF
page 1.

Reference-data area:
- Referência dos dados
- NIF

Section I — Identificação da unidade estatística:
- Número de identificação fiscal (NIF)
- Homepage
- Designação social
- Distrito/Ilha
- Município
- Freguesia
- Endereço
- Localidade
- Código postal
- Telefone
- Fax
- e-mail

Section II — Situação da unidade estatística no período de referência
dos dados:
- Situação na atividade
- Aguarda início de atividade
- Em atividade
- Atividade suspensa em
- Atividade cessada em
- Nº dias de atividade no período de referência
- Atividade económica principal (CAE Rev. 3)
- Ocorreu algum facto relevante no período de referência dos dados?
- Indique qual
- Data

Section III — Observações:
- Observações

Section IV — Responsável pelo preenchimento:
- Nome contacto
- Telefone
- Fax
- e-mail
- Função
- Assinatura
- Data

Blank boxes, blank lines and empty response areas represent fields.
Do not treat them as missing respondent observations.

Where a printed questionnaire code is represented as associated with a
target field, preserve that code exactly in Code.

Use null for Code when no printed questionnaire code is represented.

Do not infer a code from a neighbouring field or from external
knowledge.


3. UAE template elements

Physical PDF page 2 contains repeated Unidade de Atividade Económica
(UAE) blocks.

Treat the repeated blocks as multiple source instances of one reusable
template.

Extract each distinct reusable template element once:

- Código da UAE
- Designação da UAE
- Situação da UAE perante a atividade
- Observações da UAE
- Confirmar
- Produtos

Do not create duplicate records merely because the same UAE template is
represented repeatedly.

Classify these records as:

Category = "UAE template element"
Section = "UAE information"


4. Product table fields

From the product-entry table represented from physical PDF page 3,
extract one record for each of these structural fields:

- NIF
- UAE
- Período de Referência
- Nº
- Produto
- Unid.
- Código
- Quantidades produzidas
- Quantidades vendidas
- Valor das vendas / prestação de serviços
- Observações empresa
- Observações INE

Preserve the represented table-column relationships.

The rows labelled Produto a, Produto b, Produto c and Produto d are
sample/template examples. Do not extract them as respondent
observations or additional product records.

Do not extract their sample product codes or sample unit letters as
separate observations.


5. Instructions and explanatory definitions

Extract the principal predefined instructional and explanatory concepts
represented from physical PDF pages 2 and 4.

From the UAE information on page 2:
- Unidade de Atividade Económica (UAE)

From the filling instructions on page 4:
- Questionnaire scope
- Unidade monetária
- Arredondamentos
- Exemplo de arredondamento

From the explanatory notes on page 4:
- Empresa
- Unidade de Atividade Económica (UAE)
- Produtos
- Quantidades produzidas
- Quantidades vendidas
- Valor das vendas / prestação de serviços

For each instruction or definition, provide a concise source-grounded
Description preserving the substantive represented rule or definition.

Do not split supporting sentences into additional records outside this
predefined scope.


Field rules:

Category:
- Use exactly one of the five Category labels defined above.

Section:
- Identify the source section or structural region containing the
  represented element.
- Use concise stable section labels.

Field or Concept:
- Use the predefined field or concept label corresponding to the item
  being extracted.
- Preserve Portuguese source wording where the item is a visible source
  label.

Description:
- Provide a concise source-grounded description of the represented
  field, metadata item, template element, table field or instruction.
- Do not add external interpretation.
- For instructions and definitions, retain the substantive meaning
  explicitly represented by the source.

Code:
- Preserve a printed questionnaire code only when represented as
  associated with the target element.
- Use null when no printed code applies.
- Do not infer or transfer codes between neighbouring elements.

Expected Value Type:
- Assign a concise structural expected-value type based only on the
  represented questionnaire element or semantic role of the target item.
- Use consistent labels such as:
  text
  identifier
  numeric identifier
  date
  URL
  email
  telephone number
  reference period
  postal code
  category
  checkbox
  integer
  CAE code
  yes or no
  free text
  signature
  button or action
  month or period
  unit
  product code
  numeric quantity
  monetary value
  instruction
  definition
  example
- Do not infer actual respondent values.

Source Location:
- Use physical PDF page references grounded in the Markdown page
  boundaries.
- Include a concise source region or section.
- Examples include:
  "PDF page 1 — Header"
  "PDF page 1 — Reference data"
  "PDF page 1 — Section I"
  "PDF page 1 — Section II"
  "PDF page 1 — Section III"
  "PDF page 1 — Section IV"
  "PDF page 2 — UAE information"
  "PDF page 3 — Product production table"
  "PDF page 4 — Filling instructions"
  "PDF page 4 — Explanatory notes"


Additional extraction rules:

- Use only information explicitly represented in the attached
  structural Markdown.
- Preserve Portuguese wording, labels and printed codes where relevant.
- Do not invent respondent answers from blank response fields.
- Do not treat blank response areas as null observations.
- Do not duplicate repeated UAE template elements.
- Do not treat sample product rows as respondent observations.
- Do not calculate, infer, derive, repair or introduce information.
- Do not use external knowledge.
- Do not follow external links.
- Do not reconstruct hidden form data.
- Do not create records outside the predefined extraction scope.
- Ignore synthetic Markdown block labels and bounding-box metadata
  except as structural cues.
- Verify that every item within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D10",
  "branch": "B",
  "records": [
    {
      "Category": null,
      "Section": null,
      "Field or Concept": null,
      "Description": null,
      "Code": null,
      "Expected Value Type": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print("Prompt saved:", PROMPT_PATH.name)
print("Prompt SHA-256:", PROMPT_SHA256)
print(BRANCH_B_PROMPT)


In [ ]:
# ============================================================
# 6. Create pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "conversion_method":
        CONVERSION_METHOD,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "scope_filtering_applied": False,
    "page_boundaries_made_explicit": True,
    "source_block_geometry_preserved_as_metadata": True,
    "repeated_uae_blocks_retained": True,
    "sample_product_rows_retained": True,
    "ocr_applied_for_model_input": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "table_value_reconstruction_applied": False,
    "form_value_reconstruction_applied": False,
    "source_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "content_validation_performed": False,
    "source_diagnostics_file": SOURCE_DIAGNOSTICS_PATH.name,
    "source_block_audit_file": SOURCE_BLOCK_AUDIT_PATH.name,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_METADATA_PRE, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 7. Download Branch B model-input artefacts
# ============================================================

for path in [
    SOURCE_DIAGNOSTICS_PATH,
    SOURCE_BLOCK_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D10_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D10_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF, Stage 1 reference dataset, "
    "Branch A outputs, Validation A outputs, or expected counts/code mappings.\n"
    "5. Save the first complete response exactly as returned as TXT.\n"
    "6. Do not correct, repair, reorder, deduplicate, or regenerate the response."
)


In [ ]:
# ============================================================
# 8. Upload and preserve the untouched model response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D10 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = UPLOADED_RAW_RESPONSE_PATH.read_text(
    encoding="utf-8"
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError("The uploaded D10 Branch B response is empty.")

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print("Raw response preserved:", RAW_RESPONSE_PATH.name)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 9. Parse without repairing the model response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(RAW_RESPONSE_TEXT)
    valid_json = True
except json.JSONDecodeError as error:
    json_parsing_error = str(error)

top_level_object_valid = (
    valid_json
    and isinstance(parsed_response, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get("records"),
        list
    )
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 10. Schema, type and content/scope diagnostics
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(extracted_records):

        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Record is not a JSON object"
            })
            continue

        observed_fields = list(record.keys())

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Field names or field order differ",
                "expected_fields": EXPECTED_FIELDS,
                "observed_fields": observed_fields
            })

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):
                field_type_issues.append({
                    "record_index": record_index,
                    "field": field,
                    "observed_type": type(value).__name__,
                    "expected_type": "string or null"
                })

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(field)

            if (
                value is None
                or (
                    isinstance(value, str)
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index": record_index,
                    "field": field
                })


records_with_structure_issues = (
    len({
        issue["record_index"]
        for issue in record_structure_issues
    })
    if records_evaluable
    else None
)

record_schema_valid = (
    records_with_structure_issues == 0
    if records_evaluable
    else None
)

records_with_type_issues = (
    len({
        issue["record_index"]
        for issue in field_type_issues
    })
    if records_evaluable
    else None
)

field_types_valid = (
    records_with_type_issues == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)


# ------------------------------------------------------------
# Content/scope diagnostics — NOT part of structure_valid
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_records = [
        list(key)
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]

    duplicate_complete_record_count = len(
        duplicate_records
    )

    observed_non_null_codes = [
        record.get("Code")
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Code") is not None
        )
    ]

    observed_code_set = set(
        observed_non_null_codes
    )

    questionnaire_code_set_valid = (
        observed_code_set == EXPECTED_QUESTIONNAIRE_CODES
    )

    # Sample product examples should remain source content but
    # should not become extracted records.
    sample_product_pattern = re.compile(
        r"^(Produto [abcd]|[abcd]|111111111111)$",
        flags=re.IGNORECASE
    )

    sample_product_record_indices = []

    for record_index, record in enumerate(extracted_records):

        if not isinstance(record, dict):
            continue

        values_to_check = [
            record.get("Field or Concept"),
            record.get("Description")
        ]

        if any(
            isinstance(value, str)
            and sample_product_pattern.fullmatch(value.strip())
            for value in values_to_check
        ):
            sample_product_record_indices.append(
                record_index
            )

    sample_product_rows_excluded = (
        len(sample_product_record_indices) == 0
    )

    # UAE template elements must appear once each despite the
    # repeated source instances.
    expected_uae_template_fields = {
        "Código da UAE",
        "Designação da UAE",
        "Situação da UAE perante a atividade",
        "Observações da UAE",
        "Confirmar",
        "Produtos"
    }

    uae_template_records = [
        record
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category")
                == "UAE template element"
        )
    ]

    uae_template_counts = Counter(
        record.get("Field or Concept")
        for record in uae_template_records
    )

    uae_template_complete = (
        set(uae_template_counts)
        == expected_uae_template_fields
    )

    uae_template_not_duplicated = all(
        uae_template_counts.get(field, 0) == 1
        for field in expected_uae_template_fields
    )

    reference_period_field_present = any(
        isinstance(record, dict)
        and record.get("Category") == "Questionnaire field"
        and record.get("Field or Concept") == "Referência dos dados"
        for record in extracted_records
    )

    expected_product_table_fields = {
        "NIF",
        "UAE",
        "Período de Referência",
        "Nº",
        "Produto",
        "Unid.",
        "Código",
        "Quantidades produzidas",
        "Quantidades vendidas",
        "Valor das vendas / prestação de serviços",
        "Observações empresa",
        "Observações INE"
    }

    observed_product_table_fields = {
        record.get("Field or Concept")
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category") == "Product table field"
        )
    }

    product_table_scope_complete = (
        observed_product_table_fields
        == expected_product_table_fields
    )

else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_records = None
    duplicate_complete_record_count = None
    observed_non_null_codes = None
    observed_code_set = None
    questionnaire_code_set_valid = None
    sample_product_record_indices = None
    sample_product_rows_excluded = None
    uae_template_counts = None
    uae_template_complete = None
    uae_template_not_duplicated = None
    reference_period_field_present = None
    observed_product_table_fields = None
    product_table_scope_complete = None


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        observed_record_count,
    "record_count_matches_reference":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "categories_valid":
        categories_valid,
    "category_counts_match_reference":
        category_counts_valid,
    "mandatory_fields_complete":
        mandatory_fields_complete,
    "missing_mandatory_value_count":
        (
            len(missing_mandatory_values)
            if records_evaluable
            else None
        ),
    "duplicate_complete_record_count":
        duplicate_complete_record_count,
    "expected_questionnaire_codes":
        sorted(EXPECTED_QUESTIONNAIRE_CODES),
    "observed_non_null_codes":
        observed_non_null_codes,
    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,
    "reference_period_field_present":
        reference_period_field_present,
    "uae_template_complete":
        uae_template_complete,
    "uae_template_not_duplicated":
        uae_template_not_duplicated,
    "product_table_scope_complete":
        product_table_scope_complete,
    "sample_product_rows_excluded":
        sample_product_rows_excluded
}

print(json.dumps(CONTENT_DIAGNOSTICS, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 11. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "missing_mandatory_values":
        missing_mandatory_values
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 12. Preserve parsed extraction if structurally evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),

        "branch":
            parsed_response.get("branch"),

        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH
    )

else:
    print(
        "No parsed extraction created because "
        "the output is not structurally evaluable."
    )

In [ ]:
# ============================================================
# 13. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_file": RAW_RESPONSE_PATH.name,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),
    "parsed_extraction_sha256":
        parsed_extraction_sha256,
    "json_valid": valid_json,
    "records_evaluable": records_evaluable,
    "observed_record_count": observed_record_count,
    "observed_category_counts": observed_category_counts,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,
    "notes": (
        "Branch B converts the complete four-page D10 PDF to a "
        "page-aware layout-aware structural Markdown representation "
        "derived from native text blocks and source coordinates. "
        "Every non-empty source text block is retained. The repeated "
        "UAE blocks and sample product rows remain in the model "
        "representation; their scope treatment is controlled by the "
        "fixed extraction task. No OCR, semantic rewriting, "
        "normalisation, respondent-value inference or manual correction "
        "is applied. Stage 1 expected counts and code mappings are used "
        "only after extraction for diagnostics. Content-level validation "
        "is performed separately in Validation B — D10."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_match": category_counts_valid,
    "valid_json": valid_json,
    "records_evaluable": records_evaluable,
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),
    "scope_complete": record_count_valid,
    "duplicate_complete_record_count":
        duplicate_complete_record_count,
    "questionnaire_code_set_valid":
        questionnaire_code_set_valid,
    "reference_period_field_present":
        reference_period_field_present,
    "uae_template_complete":
        uae_template_complete,
    "uae_template_not_duplicated":
        uae_template_not_duplicated,
    "product_table_scope_complete":
        product_table_scope_complete,
    "sample_product_rows_excluded":
        sample_product_rows_excluded,
    "parsed_extraction_created":
        bool(structurally_evaluable),
    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D10."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 14. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    SOURCE_DIAGNOSTICS_PATH,
    SOURCE_BLOCK_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(
        PARSED_EXTRACTION_PATH
    )

print("Generated D10 Branch B files:")

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

for path in GENERATED_OUTPUTS:
    files.download(path)